# Burgers' equation

So far in this module, we have learned about the finite-difference solution for the linear and non-linear convection equations and the diffusion equation. It's time to combine all these into one: *Burgers' equation*.

Burgers' equation in one spatial dimension looks like this:

$$
\label{eq-1d-burgers-pde}
\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = \nu \frac{\partial ^2u}{\partial x^2}
$$

Here, $\nu>0$ is the viscosity. The nonlinear convection term transports the velocity field, while the diffusion term smooths its spatial variations.

## Convection and diffusion in one problem

In the convection problems, information traveled across the domain. In the diffusion problem, spatial variations became smoother. Burgers' equation lets these two processes act together: the local velocity $u$ determines how fast the field is transported, while viscosity $\nu$ smooths its gradients. A region with larger velocity can catch a slower region ahead of it, steepening the profile as diffusion works to smooth it.

Before investigating the formation of a steep front, we need a calculation whose answer we can check. We will begin with a smooth, periodic solution that is known exactly. It travels around the domain and gradually smooths, giving us a reference against which to measure numerical error.

We will also return to a question from [Lesson 6](./06-1d-convection.ipynb): does discretizing $u u_x$ give the same numerical behavior as discretizing $(u^2/2)_x$? The two expressions agree for a smooth function, but their discrete updates need not preserve the same properties. Here we will look for **preservation of the mean velocity** and **agreement with the exact profile**. A computation can pass one check without passing the other.

## Periodic domain

Imagine that the right end of the interval connects back to its left end. A profile leaving one side reappears on the other. The periodic boundary condition is

$$
\label{eq-burgers-periodic-boundary}
u(x+L,t)=u(x,t).
$$

There is no separate inlet or outlet. For a smooth periodic Burgers solution, the spatial mean stays constant: integrating the conservative form of the equation around the domain leaves no net convective or diffusive boundary flux.

We store each physical point just once. With $N$ points, the spacing is $\Delta x=L/N$, and the final stored point is $L-\Delta x$. The point at $x=L$ coincides with $x=0$ and is excluded using `endpoint=False`. This differs from our earlier grids with two distinct fixed endpoints.

## Reference solution

Let $k=2\pi/L$, $q(t)=a e^{-\nu k^2t}$, and $\theta=k(x-Ut)$. With these parameters, our reference solution is

$$\phi=1+q\cos\theta,\qquad
u(x,t)=U-2\nu\frac{\phi_x}{\phi}
=U+\frac{2\nu k q\sin\theta}{1+q\cos\theta}.$$

Here **$U$ is the mean velocity**, while $u(x,t)$ is the evolving field. Taking $0<a<1$ keeps the denominator positive. The expression is periodic and satisfies the viscous Burgers equation. Its shape translates at speed $U$, and its variation about $U$ decays.

The parameter $a$ controls the initial shape. It is dimensionless; $U$ has units of velocity, and $\nu$ has units of length squared per time. The factors $kx$ and $\nu k^2t$ are therefore dimensionless.

This expression is more complicated than the sine-decay solution in Lesson 8, but it has a similar role: it supplies an independent reference at any position and time. We will evaluate the reference solution directly in a small function. We use $L=2\pi$, $U=1$, $\nu=0.1$, and $a=0.95$. We will inspect the resulting initial profile before choosing the convection discretization.

The initial condition for this benchmark is the same expression evaluated at $t=0$. Note that changing viscosity in this formula also changes that initial profile.

First import NumPy and Matplotlib, choose the parameters, and construct the unique periodic grid. The four observation times will let us follow the profile through almost one circuit of the domain.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

L = 2.0 * np.pi
U = 1.0
nu = 0.1
a = 0.95
nx = 256
plot_times = [0.0, 1.0, 3.0, 6.0]
safety = 0.8

k = 2.0 * np.pi / L
dx = L / nx
x = np.linspace(0.0, L, nx, endpoint=False)

### Evaluate the exact solution

The function `exact_solution()` implements the formula above. It accepts an array of positions, so NumPy evaluates the entire profile at once. We keep the parameters explicit: the same function will provide the initial array and the reference at each later time.

In [ ]:
def exact_solution(x, t, nu, U, a, L):
    '''Evaluate the smooth periodic Burgers solution.'''
    k = 2.0 * np.pi / L
    q = a * np.exp(-nu * k**2 * t)
    theta = k * (x - U * t)
    return U + 2.0 * nu * k * q * np.sin(theta) / (
        1.0 + q * np.cos(theta)
    )


### Inspect the initial profile

Evaluate the exact solution at $t=0$ to construct `u0`, then plot it to see the data we will evolve. First check that $0<a<1$ and that viscosity is positive, as assumed in the reference formula.

:::{note} Python refresher — assertions
:icon: false
`assert condition` checks whether a condition is true. If it is false, Python raises an `AssertionError` and stops that cell. If it is true, execution continues. Each assertion checks only the condition it states; it does not establish that the whole calculation is correct.

Later, `np.testing.assert_allclose(actual, expected)` will check agreement within a numerical tolerance. This allows for the small rounding differences of floating-point arithmetic.
:::


In [ ]:
assert 0.0 < a < 1.0
assert nu > 0.0

u0 = exact_solution(x, 0.0, nu, U, a, L)
print(f'Initial velocity range: {u0.min():.3f} to {u0.max():.3f}')
print(f'One circuit at mean speed U takes L/U = {L/U:.3f}')

For plotting only, append the point at $x=L$ with the value at $x=0$ to close the periodic curve. The solver will still use the original array of unique points.

In [ ]:
fig_initial, ax_initial = plt.subplots(figsize=(5.0, 3.0))
ax_initial.plot(
    np.append(x, L), np.append(u0, u0[0]),
    color='#181712', linewidth=1.8,
)
ax_initial.axhline(0.0, color='#787876', linestyle=':', linewidth=1)
ax_initial.set(xlabel='x', ylabel='u', title='Initial velocity profile',
               xlim=(0.0, L), ylim=(-0.1, 1.8))
ax_initial.grid(alpha=0.25)
fig_initial.tight_layout()
plt.show()

The profile lies above zero; its smallest sampled velocity is approximately $0.392$. Confirm this at every grid point with `np.all(u0 > 0.0)`: the comparison produces one Boolean value per point, and `np.all` returns true only if every comparison is true.

These positive initial velocities give us the setting for backward differences in convection. Later, the time-step argument will show why both numerical updates keep their values within the initial range, preserving positivity throughout the calculation.

In [ ]:
assert np.all(u0 > 0.0)

## Two ways to discretize convection

Write $\lambda=\Delta t/\Delta x$ and $r=\nu\Delta t/\Delta x^2$. The original pointwise update is

$$u_i^{n+1}=u_i^n-\lambda u_i^n(u_i^n-u_{i-1}^n)
+r(u_{i+1}^n-2u_i^n+u_{i-1}^n).$$

For a smooth solution, $u u_x=(u^2/2)_x$. Discretizing this flux derivative instead gives the conservative update:

$$u_i^{n+1}=u_i^n-\lambda\left[f(u_i^n)-f(u_{i-1}^n)\right]
+r(u_{i+1}^n-2u_i^n+u_{i-1}^n),\qquad f(u)=\frac{u^2}{2}.$$

`np.roll(u, 1)` supplies every left neighbor, including across the periodic seam; `np.roll(u, -1)` supplies every right neighbor. All terms read the old state. Backward flux differencing here is used only for positive velocities.



:::{note} Python refresher — periodic neighbors with `np.roll`
:icon: false
`np.roll(array, shift)` shifts the entries and wraps those that pass an end back to the other side. For example, with `u = np.array([10, 20, 30, 40])`:

```python
np.roll(u, 1)   # [40, 10, 20, 30]: the left neighbor of each point
np.roll(u, -1)  # [20, 30, 40, 10]: the right neighbor of each point
```

The returned arrays align each neighbor with the point being updated. At the first point, the left neighbor is the last stored value. `np.roll` does not change `u`, and our step functions form a new array, so every term uses the same old time level.
:::

### The pointwise update

`burgers_step()` combines the backward difference of $u$ multiplied by the local velocity with the centered second difference. We retain it as the comparison method from our earlier convection work.

In [ ]:
def burgers_step(u, dt, dx, nu):
    '''Take one periodic pointwise-convection/centered-diffusion step.'''
    left = np.roll(u, 1)
    right = np.roll(u, -1)
    return (
        u - dt / dx * u * (u - left)
        + nu * dt / dx**2 * (right - 2.0 * u + left)
    )


### The conservative update

The function `conservative_burgers_step()` changes only the convection term: it subtracts neighboring values of the flux $f(u)=u^2/2$. The diffusion term and periodic neighbors are identical. This isolates the effect of the convection discretization.

In [ ]:
def conservative_burgers_step(u, dt, dx, nu):
    '''Take one periodic conservative-convection/centered-diffusion step.'''
    left = np.roll(u, 1)
    right = np.roll(u, -1)
    return (
        u - dt / dx * (0.5 * u**2 - 0.5 * left**2)
        + nu * dt / dx**2 * (right - 2.0 * u + left)
    )

### Why the conservative update preserves the mean

Define $\bar u^n=N^{-1}\sum_i u_i^n$, counting each periodic point once. Summing the conservative update gives

$$\bar u^{n+1}-\bar u^n=
-\frac{\lambda}{N}\sum_i[f(u_i^n)-f(u_{i-1}^n)]
+\frac{r}{N}\sum_i[u_{i+1}^n-2u_i^n+u_{i-1}^n]=0.$$

Each shifted sum contains exactly the same values: both differences telescope around the periodic grid. Conservation follows from the discrete algorithm, up to floating-point roundoff.

For the pointwise scheme, the identity

$$\sum_i u_i(u_i-u_{i-1})=\frac12\sum_i(u_i-u_{i-1})^2$$

instead gives

$$\bar u^{n+1}-\bar u^n=-\frac{\lambda}{2N}\sum_i(u_i^n-u_{i-1}^n)^2.$$

It loses mean whenever the old state is nonconstant. This is a property of the discretization, not an indexing error.



### Derive a common time-step bound

For the conservative scheme, factor the flux difference and define

$$b_i=\frac{\lambda}{2}(u_i^n+u_{i-1}^n).$$

Then

$$u_i^{n+1}=(b_i+r)u_{i-1}^n+(1-b_i-2r)u_i^n+r u_{i+1}^n.$$

If $0\le u_i^n\le M$, then $0\le b_i\le\lambda M$. Thus all three weights are nonnegative and sum to one whenever

$$\Delta t\left(\frac{M}{\Delta x}+\frac{2\nu}{\Delta x^2}\right)\le1.$$

The pointwise scheme has the same weights with $b_i$ replaced by $\lambda u_i^n$, so this sufficient bound applies to both. Each new value stays within the old range. By induction we may use $M=\max_i u_i^0$ throughout, including preservation of positive velocities.

This is a range-preservation argument, not a general nonlinear perturbation-stability proof. Conservation and range preservation alone do not establish accuracy.

## Check one update before integrating

Use the small periodic array $[1,2,1]$ with $\Delta t=0.1$, $\Delta x=1$, and $\nu=0.1$. Its left neighbors are $[1,1,2]$ and its right neighbors are $[2,1,1]$. Substitution into the two updates gives $[1.01,1.78,1.11]$ for the pointwise scheme and $[1.01,1.83,1.16]$ for the conservative scheme.

The conservative values sum to the initial total of $4$; the pointwise values sum to $3.9$. The assertions compare the functions against these independently calculated values. We also check that the input array is unchanged and that both schemes preserve a constant state, for which every spatial difference vanishes.

In [ ]:
# Hand-checkable periodic update: left neighbors are [1, 1, 2].
u_test = np.array([1.0, 2.0, 1.0])
np.testing.assert_allclose(
    burgers_step(u_test, dt=0.1, dx=1.0, nu=0.1),
    [1.01, 1.78, 1.11],
)
np.testing.assert_allclose(
    conservative_burgers_step(u_test, dt=0.1, dx=1.0, nu=0.1),
    [1.01, 1.83, 1.16],
)
np.testing.assert_array_equal(u_test, [1.0, 2.0, 1.0])
for step in (burgers_step, conservative_burgers_step):
    constant = np.full(5, 1.5)
    np.testing.assert_allclose(step(constant, 0.1, 1.0, 0.1), constant)
print('One-step values, unchanged input, and constant-state checks pass.')

## Integrate both methods to the same times

Use the initial maximum $M=\max_i u_i^0$ in the time-step bound and multiply the permitted step by the safety factor $0.8$. The range-preservation argument lets us use this bound throughout the run.

For each observation interval, `np.ceil` rounds the required number of steps upward. Dividing the interval by that count gives a step no larger than the target and lands exactly at the requested time. Both methods receive that same step.

The dictionaries `states` and `snapshots` keep the two calculations separate, with each starting from its own copy of `u0`. We retain only the four requested profiles. Within the loop, assertions check for nonfinite values and departures from the initial range. We also record the largest mean drift over all steps and check that the conservative scheme's drift stays below $10^{-12}$ for this run.

In [ ]:
# Both methods start independently and use exactly the same time steps.
assert 0.0 < safety <= 1.0
dt_target = safety / (u0.max() / dx + 2.0 * nu / dx**2)
methods = {
    'Pointwise': burgers_step,
    'Conservative': conservative_burgers_step,
}
states = {name: u0.copy() for name in methods}
snapshots = {name: [u0.copy()] for name in methods}
step_counts = [0]
time = 0.0
total_steps = 0
max_mean_drift = {name: 0.0 for name in methods}

for target_time in plot_times[1:]:
    num_steps = int(np.ceil((target_time - time) / dt_target))
    dt = (target_time - time) / num_steps
    for _ in range(num_steps):
        for name, step in methods.items():
            states[name] = step(states[name], dt, dx, nu)
            actual = states[name]
            assert np.all(np.isfinite(actual))
            assert actual.min() >= u0.min() - 1e-12
            assert actual.max() <= u0.max() + 1e-12
            max_mean_drift[name] = max(
                max_mean_drift[name], abs(actual.mean() - u0.mean())
            )
    total_steps += num_steps
    for name in methods:
        snapshots[name].append(states[name].copy())
    step_counts.append(total_steps)
    time = target_time

assert max_mean_drift['Conservative'] < 1e-12
print(f'Target time step: {dt_target:.6f}')
print(f'Total steps per method to t={time:g}: {total_steps}')
print('Maximum absolute mean drift over all steps:', max_mean_drift)

## Compare the profiles

The extra point at $x=L$ below is added **only for plotting**, by copying the value at $x=0$. The numerical solver always uses the unique periodic grid.

All panels share axis limits, so changes in amplitude can be compared directly. The exact solution is drawn in ink, the pointwise result in gray, and the conservative result in red. Line styles distinguish the curves as well as color.

In [ ]:
x_plot = np.append(x, L)
fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.0), sharex=True, sharey=True)
styles = {'Pointwise': ('#787876', '-'), 'Conservative': ('#ce2406', '-.')}

for j, (ax, t) in enumerate(zip(axes.flat, plot_times)):
    reference = exact_solution(x, t, nu, U, a, L)
    ax.plot(x_plot, np.append(reference, reference[0]),
            color='#181712', linestyle='--', linewidth=2, label='Exact')
    for name, (color, linestyle) in styles.items():
        numerical = snapshots[name][j]
        ax.plot(x_plot, np.append(numerical, numerical[0]),
                color=color, linestyle=linestyle, linewidth=1.7, label=name)
    ax.axhline(U, color='#b4b4b2', linewidth=0.8)
    ax.set_title(f't = {t:g}')
    ax.set_xlim(0.0, L)
    ax.grid(alpha=0.25)

axes[0, 0].legend(fontsize=8)
for ax in axes[-1, :]:
    ax.set_xlabel('x')
for ax in axes[:, 0]:
    ax.set_ylabel('u')
fig.suptitle('Smooth periodic Burgers solution', fontsize=12)
fig.tight_layout()
plt.show()

## Measure what the plot cannot establish

The curves are close enough that visual agreement alone is difficult to judge. Compute the maximum error over the grid and the change in the discrete mean:

$$
\label{eq-burgers-comparison-diagnostics}
E_\infty(t)=\max_i|u_i^{\mathrm{computed}}-u(x_i,t)|,
\qquad
\Delta\bar u(t)=\frac1N\sum_i u_i^{\mathrm{computed}}-\frac1N\sum_i u_i^0.
$$

The first quantity measures agreement with the exact reference. The second measures whether the algorithm preserves its initial discrete mean; it uses the sampled initial mean rather than assuming that a finite grid integrates the exact profile perfectly. Neither diagnostic replaces the other.

In [ ]:
print(f"{'Time':>5} {'Method':>13} {'Steps':>7} "
      f"{'Maximum error':>16} {'Mean drift':>16}")
for j, (t, steps) in enumerate(zip(plot_times, step_counts)):
    reference = exact_solution(x, t, nu, U, a, L)
    for name in methods:
        numerical = snapshots[name][j]
        error = np.max(np.abs(numerical - reference))
        drift = numerical.mean() - u0.mean()
        print(f'{t:5.1f} {name:>13} {steps:7d} '
              f'{error:16.6e} {drift:16.6e}')

At $t=6$, the maximum error is about $0.0216$ for the pointwise scheme and $0.0101$ for the conservative scheme. The pointwise mean has decreased by about $0.00453$, while the conservative mean is preserved to roundoff. The negative drift agrees with the sum-of-squares expression derived above.

For these parameters and this grid, the conservative scheme both preserves the mean and has a smaller maximum error. Conservation does not guarantee a smaller error in every comparison, and a single grid does not establish convergence. We have checked an important discrete property and measured accuracy for one smooth problem.

:::{warning .simple .dropdown icon=false open=false} Self-checks
- Why does periodicity make the conservative flux differences sum to zero?
- Both schemes preserve a constant state. Why is that check insufficient to distinguish them?
- Does preserving the mean imply that the computed profile has the correct shape? Use the two columns of the table to support your answer.
:::
